# 手撕-Attention

In [23]:
import torch
from torch import nn
import torch.nn.functional as F
import math

## 单头Cross Attention

这部分不是Transformer用的，非自注意力机制，只是一个“开胃小菜”

- transpose()的语法

- 使用nn.Linear而不是定义权重矩阵

- nn.Linear需要指定bias=False

- softmax的调用需要指定dim

- 约定俗成的风格：不需要学习的层使用torch.nn.functional

- 使用math.sqrt(常量)而不是torch.sqrt(torch.tensor(常量))，因为前者是python标量，后者是会默认放在CPU上的张量

In [ ]:
class SingleHeadCrossAttention(nn.Module):
    def __init__(self, query_dim, kv_dim, head_dim):
        super().__init__()
        self.head_dim = head_dim
        self.q_proj = nn.Linear(query_dim, head_dim, bias=False)
        self.k_proj = nn.Linear(kv_dim, head_dim, bias=False)
        self.v_proj = nn.Linear(kv_dim, head_dim, bias=False)

    def forward(self, query_source, kv_source):
        # query_source: (batch_size, seq_len, query_dim); kv_source: (batch_size, seq_len, kv_dim)
        Q = self.q_proj(query_source) # (batch_size, seq_len, head_dim)
        K = self.k_proj(kv_source) # (batch_size, seq_len, head_dim)
        V = self.v_proj(kv_source) # (batch_size, seq_len, head_dim)

        values = torch.matmul(Q,K.transpose(1,2)) # (batch_size, seq_len, seq_len), [batch_num, i, j]表示序列中第i个token的Q查询第j个token的K的得分
        scale = math.sqrt(K.size(-1)) # 不占用GPU的标量
        
        scores = values/scale
        weights = F.softmax(scores, dim=-1) # (batch_size, seq_len, seq_len) 不要忘了dim=-1!!!
        attention = torch.matmul(weights, V) # (batch_size, seq_len, head_dim)
        return attention, weights

In [25]:
# 定义模型参数
BATCH_SIZE = 4
QUERY_SEQ_LEN = 10  # 假设是解码器已生成的序列长度
KV_SEQ_LEN = 15     # 假设是编码器输出的序列长度
QUERY_DIM = 512     # 解码器隐藏状态维度
KV_DIM = 768        # 编码器输出维度 (可以和query_dim不同)
HEAD_DIM = 64       # 注意力头的维度

# 1. 创建两个随机的、不同来源的输入张量
# 模拟解码器的输入
query_input = torch.randn(BATCH_SIZE, QUERY_SEQ_LEN, QUERY_DIM)
# 模拟编码器的输出
kv_input = torch.randn(BATCH_SIZE, KV_SEQ_LEN, KV_DIM)

print("--- 输入张量 ---")
print(f"Query源的形状: {query_input.shape}")
print(f"K/V源的形状  : {kv_input.shape}\n")

# 2. 实例化我们的交叉注意力模块
cross_attention_module = SingleHeadCrossAttention(
    query_dim=QUERY_DIM,
    kv_dim=KV_DIM,
    head_dim=HEAD_DIM
)
print("单头交叉注意力模块已创建\n")

# 3. 将输入送入模块进行前向传播
output, attention_weights = cross_attention_module(query_input, kv_input)

# 4. 检查输出的形状
print("--- 输出结果 ---")
print(f"最终输出的形状: {output.shape}")
print(f"预期输出形状: ({BATCH_SIZE}, {QUERY_SEQ_LEN}, {HEAD_DIM})\n")

print(f"注意力权重的形状: {attention_weights.shape}")
print(f"预期权重形状: ({BATCH_SIZE}, {QUERY_SEQ_LEN}, {KV_SEQ_LEN})\n")

# 检查权重矩阵，确认每一行和为1
first_sample_weights = attention_weights[0]
sum_of_weights_first_row = first_sample_weights[0].sum()
print(f"第一个样本中，Query序列第一个词 对 K/V序列所有词 的注意力权重之和: {sum_of_weights_first_row:.4f}")
assert torch.allclose(sum_of_weights_first_row, torch.tensor(1.0))
print("权重和检查通过！")

--- 输入张量 ---
Query源的形状: torch.Size([4, 10, 512])
K/V源的形状  : torch.Size([4, 15, 768])

单头交叉注意力模块已创建

--- 输出结果 ---
最终输出的形状: torch.Size([4, 10, 64])
预期输出形状: (4, 10, 64)

注意力权重的形状: torch.Size([4, 10, 15])
预期权重形状: (4, 10, 15)

第一个样本中，Query序列第一个词 对 K/V序列所有词 的注意力权重之和: 1.0000
权重和检查通过！


## 单头Self-Attention

In [35]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=True)
    
    def forward(self, x): # (batch_size, seq_len, input_dim)
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        scores = torch.matmul(Q,K.transpose(1,2))
        scale = math.sqrt(K.size(-1))
        scores = scores/scale

        weights = F.softmax(scores, dim=-1)
        attention = torch.matmul(weights, V)
        return attention, weights

In [36]:
# --- 验证逻辑部分 ---

print("--- 开始验证 SingleHeadSelfAttention 模块 ---\n")

# 1. 定义超参数
BATCH_SIZE = 4
SEQ_LEN = 10
EMBED_DIM = 128


print(f"设定参数: Batch Size={BATCH_SIZE}, Seq Len={SEQ_LEN}, Embed Dim={EMBED_DIM}\n")

# 2. 实例化模型
# (假设 SingleHeadSelfAttention 类已经在上面的单元格中定义好了)
model = SingleHeadSelfAttention(embed_dim=EMBED_DIM)
print("模型实例化成功！\n")


# 3. 创建虚拟输入数据
dummy_input = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
print(f"创建虚拟输入，形状为: {dummy_input.shape}\n")

# 4. 执行前向传播并开始验证
try:
    print("执行前向传播并开始验证...\n")
    attention_output, attention_weights = model(dummy_input)

    # 验证点 1: 输出张量的形状是否正确
    print("1. 验证输出形状...")
    expected_output_shape = (BATCH_SIZE, SEQ_LEN, EMBED_DIM)
    actual_output_shape = attention_output.shape
    assert actual_output_shape == expected_output_shape
    print(f"✅ 验证通过! 输出形状为 {actual_output_shape}\n")

    # 验证点 2: 注意力权重矩阵的形状是否正确
    print("2. 验证注意力权重形状...")
    expected_weights_shape = (BATCH_SIZE, SEQ_LEN, SEQ_LEN)
    actual_weights_shape = attention_weights.shape
    assert actual_weights_shape == expected_weights_shape
    print(f"✅ 验证通过! 权重形状为 {actual_weights_shape}\n")

    # 验证点 3: 注意力权重是否已正确归一化 (每一行的和应为1)
    print("3. 验证注意力权重归一化...")
    weight_sums = attention_weights.sum(dim=-1)
    ones_tensor = torch.ones(BATCH_SIZE, SEQ_LEN)
    assert torch.allclose(weight_sums, ones_tensor)
    print("✅ 验证通过! 所有注意力权重行的和都接近1。\n")

    print("--- 所有验证成功！代码核心逻辑正确。 ---")

except Exception as e:
    print(f"❌ 验证过程中出现错误: {e}")

--- 开始验证 SingleHeadSelfAttention 模块 ---

设定参数: Batch Size=4, Seq Len=10, Embed Dim=128

模型实例化成功！

创建虚拟输入，形状为: torch.Size([4, 10, 128])

执行前向传播并开始验证...

1. 验证输出形状...
✅ 验证通过! 输出形状为 torch.Size([4, 10, 128])

2. 验证注意力权重形状...
✅ 验证通过! 权重形状为 torch.Size([4, 10, 10])

3. 验证注意力权重归一化...
✅ 验证通过! 所有注意力权重行的和都接近1。

--- 所有验证成功！代码核心逻辑正确。 ---


## Multi-Head Self-Attention

大致思路：

1. 先获得三组向量（连在一起的），然后用view拆分成num_heads个小的，将num_heads这一维放到seq_len前面（因为是头之间并行）

2. 最后两维的操作，和单头注意力一样，获得多个头的Attention输出

3. 将输出再view成要的格式，乘以$W_O$后得到最终结果


- 记得最后的合并矩阵以及合并操作!!!

- .contiguous(): 将张量在内存中的实际存储顺序与逻辑顺序一致, 在transpose()之后用

In [49]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, num_heads, embed_dim):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        self.o_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x): # (batch_size, seq_len, embed_dim)
        batch_size, seq_len, embed_dim = x.shape
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, -1).transpose(1,2) # (batch_size, seq_len, num_heads, head_dim)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_heads, -1).transpose(1,2)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_heads, -1).transpose(1,2)
        scores = torch.matmul(Q,K.transpose(-2,-1)) # (batch_size, num_heads, seq_len, seq_len)
        scale = math.sqrt(K.size(-1))
        scores = scores / scale
        weights = F.softmax(scores, dim=-1)
        attention = torch.matmul(weights, V) # (batch_size, num_heads, seq_len, head_dim)
        # 记得合并! .contagious()
        attention = attention.transpose(1,2).contiguous().view(batch_size,seq_len,self.embed_dim) 
        attention = self.o_proj(attention)
        return attention, weights

In [50]:
# --- 验证逻辑部分 ---

print("--- 开始验证 MultiHeadSelfAttention 模块 ---\n")

# 1. 定义超参数
BATCH_SIZE = 4
SEQ_LEN = 10
EMBED_DIM = 512
NUM_HEADS = 8

print(f"设定参数: Batch Size={BATCH_SIZE}, Seq Len={SEQ_LEN}, Embed Dim={EMBED_DIM}, Num Heads={NUM_HEADS}\n")

# 2. 实例化模型
# (假设 MultiHeadSelfAttention 类已经在上面的单元格中定义好了)
model = MultiHeadSelfAttention(embed_dim=EMBED_DIM, num_heads=NUM_HEADS)
print("模型实例化成功！\n")


# 3. 创建虚拟输入数据
dummy_input = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
print(f"创建虚拟输入，形状为: {dummy_input.shape}\n")

# 4. 执行前向传播并开始验证
try:
    print("执行前向传播并开始验证...\n")
    final_output, attention_weights = model(dummy_input)

    # 验证点 1: 最终输出张量的形状是否正确 (这是最重要的检查点)
    print("1. 验证最终输出形状...")
    expected_output_shape = (BATCH_SIZE, SEQ_LEN, EMBED_DIM)
    actual_output_shape = final_output.shape
    assert actual_output_shape == expected_output_shape
    print(f"✅ 验证通过! 输出形状为 {actual_output_shape}\n")

    # 验证点 2: 注意力权重矩阵的形状是否正确
    print("2. 验证注意力权重形状...")
    expected_weights_shape = (BATCH_SIZE, NUM_HEADS, SEQ_LEN, SEQ_LEN)
    actual_weights_shape = attention_weights.shape
    assert actual_weights_shape == expected_weights_shape
    print(f"✅ 验证通过! 权重形状为 {actual_weights_shape}\n")

    # 验证点 3: 注意力权重是否已正确归一化 (每个头的每一行的和应为1)
    print("3. 验证注意力权重归一化...")
    weight_sums = attention_weights.sum(dim=-1)
    ones_tensor = torch.ones(BATCH_SIZE, NUM_HEADS, SEQ_LEN)
    assert torch.allclose(weight_sums, ones_tensor)
    print("✅ 验证通过! 所有注意力权重行的和都接近1。\n")

    print("--- 所有验证成功！代码实现正确。 ---")

except Exception as e:
    print(f"❌ 验证过程中出现错误: {e}")

--- 开始验证 MultiHeadSelfAttention 模块 ---

设定参数: Batch Size=4, Seq Len=10, Embed Dim=512, Num Heads=8

模型实例化成功！

创建虚拟输入，形状为: torch.Size([4, 10, 512])

执行前向传播并开始验证...

1. 验证最终输出形状...
✅ 验证通过! 输出形状为 torch.Size([4, 10, 512])

2. 验证注意力权重形状...
✅ 验证通过! 权重形状为 torch.Size([4, 8, 10, 10])

3. 验证注意力权重归一化...
✅ 验证通过! 所有注意力权重行的和都接近1。

--- 所有验证成功！代码实现正确。 ---


## Masked Multi-Head Self-Attention

In [53]:
class MaskedMultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim//num_heads
        
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        self.o_proj = nn.Linear(embed_dim, embed_dim, bias=False)
    
    def forward(self, x, mask=None):
        # 先算QKV向量
        batch_size, seq_len, embed_dim = x.shape
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2) # (batch_size, num_heads, seq_len, head_dim)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)

        # 注意力权重
        scores = torch.matmul(Q,K.transpose(-2,-1))
        scale = math.sqrt(K.size(-1))
        scores = scores / scale
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))

        weights = F.softmax(scores, dim=-1) # (batch_size, num_heads, seq_len, head_dim)
        # 加权平均, 连接并输出
        attention = torch.matmul(weights, V)
        attention = attention.transpose(1,2).contiguous().view(batch_size, seq_len, embed_dim)
        attention = self.o_proj(attention)
        return attention, weights

In [54]:
# --- 验证逻辑部分 ---

print("--- 开始验证 MaskedMultiHeadSelfAttention 模块 ---\n")

# 1. 定义超参数
BATCH_SIZE = 4
SEQ_LEN = 10
EMBED_DIM = 512
NUM_HEADS = 8

print(f"设定参数: Batch Size={BATCH_SIZE}, Seq Len={SEQ_LEN}, Embed Dim={EMBED_DIM}, Num Heads={NUM_HEADS}\n")

# 2. 实例化模型
model = MaskedMultiHeadSelfAttention(embed_dim=EMBED_DIM, num_heads=NUM_HEADS)
print("带掩码的多头注意力模块已创建！\n")

# 3. 创建虚拟输入数据
dummy_input = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
print(f"创建虚拟输入，形状为: {dummy_input.shape}\n")

# --- 4. 创建因果掩码 (Causal Mask) ---
# 这是解码器自注意力的关键
# 我们需要一个 (SEQ_LEN, SEQ_LEN) 的矩阵
# 其中上三角（不含对角线）部分为 True，其他为 False
mask = torch.triu(torch.ones(SEQ_LEN, SEQ_LEN), diagonal=1).bool()
print("创建的因果掩码 (True代表需要被遮盖的位置):")
print(mask)
print("\n")


# 5. 执行前向传播，并传入掩码
try:
    print("执行前向传播并开始验证...\n")
    final_output, attention_weights = model(dummy_input, mask=mask)

    # 验证点 1: 最终输出形状 (和之前一样)
    print("1. 验证最终输出形状...")
    expected_output_shape = (BATCH_SIZE, SEQ_LEN, EMBED_DIM)
    assert final_output.shape == expected_output_shape
    print(f"✅ 验证通过! 输出形状为 {final_output.shape}\n")

    # 验证点 2: 注意力权重形状 (和之前一样)
    print("2. 验证注意力权重形状...")
    expected_weights_shape = (BATCH_SIZE, NUM_HEADS, SEQ_LEN, SEQ_LEN)
    assert attention_weights.shape == expected_weights_shape
    print(f"✅ 验证通过! 权重形状为 {attention_weights.shape}\n")

    # 【核心验证点 3】: 检查掩码是否生效
    print("3. 验证掩码是否生效...")
    # 取出第一个样本的第一个头的权重矩阵
    first_head_weights = attention_weights[0, 0]
    print("第一个样本第一个头的权重矩阵 (部分):")
    # 为了方便查看，只打印小数点后2位
    print(torch.round(first_head_weights, decimals=2))
    print("\n")

    # 检查上三角（不含对角线）的权重是否都为0
    # triu(..., diagonal=1) 会取出上三角部分
    upper_triangle_sum = torch.triu(first_head_weights, diagonal=1).sum()
    assert torch.allclose(upper_triangle_sum, torch.tensor(0.0)), "掩码未生效，上三角权重不为0！"
    print("✅ 验证通过! 注意力权重的上三角部分全为0，模型没有看到未来的信息。\n")
    
    print("--- 所有验证成功！带掩码的多头自注意力实现正确。 ---")

except Exception as e:
    print(f"❌ 验证过程中出现错误: {e}")

--- 开始验证 MaskedMultiHeadSelfAttention 模块 ---

设定参数: Batch Size=4, Seq Len=10, Embed Dim=512, Num Heads=8

带掩码的多头注意力模块已创建！

创建虚拟输入，形状为: torch.Size([4, 10, 512])

创建的因果掩码 (True代表需要被遮盖的位置):
tensor([[False,  True,  True,  True,  True,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True,  True,  True,  True,  True],
        [False, False, False, False,  True,  True,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False, False, False,  True,  True,  True,  True],
        [False, False, False, False, False, False, False,  True,  True,  True],
        [False, False, False, False, False, False, False, False,  True,  True],
        [False, False, False, False, False, False, False, False, False,  True],
        [False, False, False, False, False, False, False, False, False, False]])


执行前向传播并开始验

## GQA

关键：K和V最开始算的时候没有Q的维度大（因为切分后的数量更少），然后等要算点积注意力的时候用.repeat_interleave()来算

In [59]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, embed_dim, num_kv_heads, num_q_heads):
        super().__init__()
        assert embed_dim % num_q_heads == 0
        assert num_q_heads % num_kv_heads == 0
        self.num_kv_heads = num_kv_heads
        self.num_q_heads = num_q_heads
        self.embed_dim = embed_dim
        self.head_dim = embed_dim // num_q_heads
        

        self.q_proj = nn.Linear(embed_dim, num_q_heads*self.head_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, num_kv_heads*self.head_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, num_kv_heads*self.head_dim, bias=False)
        self.o_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x, mask=None):
        batch_size, seq_len, embed_dim = x.shape
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_q_heads, self.head_dim).transpose(1,2)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1,2)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1,2)

        repeat_factor = self.num_q_heads//self.num_kv_heads
        if repeat_factor > 1:
            K = K.repeat_interleave(repeat_factor, dim=1)
            V = V.repeat_interleave(repeat_factor, dim=1)
        
        
        score = torch.matmul(Q,K.transpose(-2,-1))
        scale = math.sqrt(K.size(-1))
        score = score/scale
        if mask is not None:
            score.masked_fill(mask, float('-inf'))
        weights = F.softmax(score, dim=-1)
        attention = torch.matmul(weights, V).transpose(1,2).contiguous().view(batch_size, seq_len, embed_dim)
        attention = self.o_proj(attention)
        return attention, weights

In [60]:
print("--- 开始验证 GroupedQueryAttention 模块 ---\n")

# 1. 定义超参数
BATCH_SIZE = 4
SEQ_LEN = 10
EMBED_DIM = 512
NUM_Q_HEADS = 8  # 8个查询头
NUM_KV_HEADS = 2 # 2个键/值头 (每4个Q头共享一对K/V)

print(f"设定参数: Embed Dim={EMBED_DIM}, Q Heads={NUM_Q_HEADS}, K/V Heads={NUM_KV_HEADS}\n")

# 2. 实例化模型
model = GroupedQueryAttention(
    embed_dim=EMBED_DIM,
    num_q_heads=NUM_Q_HEADS,
    num_kv_heads=NUM_KV_HEADS
)
print("GQA模块已创建！\n")

# 3. 创建虚拟输入数据
dummy_input = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
print(f"创建虚拟输入，形状为: {dummy_input.shape}\n")

# 4. 执行前向传播
try:
    print("执行前向传播并开始验证...\n")
    final_output, attention_weights = model(dummy_input)

    # 验证点 1: 最终输出张量的形状是否正确
    print("1. 验证最终输出形状...")
    expected_output_shape = (BATCH_SIZE, SEQ_LEN, EMBED_DIM)
    assert final_output.shape == expected_output_shape
    print(f"✅ 验证通过! 输出形状为 {final_output.shape}\n")

    # 验证点 2: 注意力权重矩阵的形状是否正确
    print("2. 验证注意力权重形状...")
    expected_weights_shape = (BATCH_SIZE, NUM_Q_HEADS, SEQ_LEN, SEQ_LEN)
    assert attention_weights.shape == expected_weights_shape
    print(f"✅ 验证通过! 权重形状为 {attention_weights.shape}\n")

    print("--- 所有验证成功！GQA实现正确。 ---")

except Exception as e:
    print(f"❌ 验证过程中出现错误: {e}")

--- 开始验证 GroupedQueryAttention 模块 ---

设定参数: Embed Dim=512, Q Heads=8, K/V Heads=2

GQA模块已创建！

创建虚拟输入，形状为: torch.Size([4, 10, 512])

执行前向传播并开始验证...

1. 验证最终输出形状...
✅ 验证通过! 输出形状为 torch.Size([4, 10, 512])

2. 验证注意力权重形状...
✅ 验证通过! 权重形状为 torch.Size([4, 8, 10, 10])

--- 所有验证成功！GQA实现正确。 ---
